# raw_to_dst — Convert raw tag CSV to standard format

Run this **once** per tag. Produces:
```
{TAG_ROOT}/{tag_name}/
    dst.csv            ← time, temperature, pressure, light
    tagging_event.csv  ← release + fish_death
    metadata.json      ← tag_name, tag_type
```
After this, use `load_tag(tag_root=TAG_ROOT, tag_name=tag_name)` normally.


In [3]:
import sys
sys.path.append("../")


TAG_ROOT            = "/tmp/tags"    

TAG_TYPE            = "wc_psat"
tag_name            = "263936"                                                                                                                                                                                                               
RAW_DATA_PATH       = "../docs/specs/reference/tuna-tristan/tuna-tristan/263936/263936-Series.csv"
TAGGING_EVENTS_PATH = "263936/tagging_events.csv"                                                                                                                                                                                  
TAG_ROOT            = ""    

scratch_root = "s3://gfts-ifremer/tuna_mediterranean/tags/formatted"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}


In [4]:
from pangeo_fish.light.ingest import prepare_tag_folder

prepare_tag_folder(
    RAW_DATA_PATH,
    TAG_TYPE,
    tagging_events_path=TAGGING_EVENTS_PATH,
    output_dir=TAG_ROOT,
    tag_name=tag_name,
)


  288 rows | 2026-03-07 00:00:00 → 2026-03-10 07:50:00
Tag folder ready: 263936


'263936'

## Verify

In [5]:
import sys
sys.path.append("../")
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(tag_root=TAG_ROOT, tag_name=tag_name)
print(f"Deployment: {time_slice.start} → {time_slice.stop}")
print(f"tag_log: {len(tag_log.time):,} timesteps")
tag


/home/ecap/micromamba/envs/pangeo-fish_debug/lib/python3.11/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


Deployment: 2024-09-07T11:05:00.000000000 → 2026-03-12T00:00:00.000000000
tag_log: 288 timesteps


<xarray.DataTree>
Group: /
│   Attributes:
│       tag_name:  263936
│       tag_type:  wc_psat
├── Group: /dst
│       Dimensions:      (time: 288)
│       Coordinates:
│         * time         (time) datetime64[ns] 2kB 2026-03-07 ... 2026-03-10T07:50:00
│       Data variables:
│           temperature  (time) float64 2kB 13.2 13.0 13.3 13.0 ... 13.5 13.5 13.5 13.5
│           pressure     (time) float64 2kB 35.5 43.5 19.5 25.0 30.0 ... 3.0 3.0 6.0 4.5
│           light        (time) float64 2kB nan nan nan nan nan ... nan nan nan nan nan
└── Group: /tagging_events
        Dimensions:     (event_name: 2)
        Coordinates:
          * event_name  (event_name) object 16B 'release' 'fish_death'
        Data variables:
            time        (event_name) datetime64[ns] 16B 2024-09-07T11:05:00 2026-03-12
            longitude   (event_name) float64 16B -1.984 -1.674
            latitude    (event_name) float64 16B 46.31 44.42

In [4]:
import s3fs

s3 = s3fs.S3FileSystem(**storage_options)
local_folder = f"{TAG_ROOT}/{tag_name}"
s3_folder    = f"{scratch_root}/{tag_name}"

for fname in ["dst.csv", "tagging_event.csv", "metadata.json"]:
    local = f"{local_folder}/{fname}"
    remote = f"{s3_folder}/{fname}"
    s3.put(local, remote)
    print(f"Uploaded → {remote}")

print(f"\nDone. Load with:")
print(f'  load_tag(tag_root=\"{scratch_root}\", tag_name=\"{tag_name}\")')


Uploaded → s3://gfts-ifremer/tuna_mediterranean/tags/formatted/281B-4949/dst.csv
Uploaded → s3://gfts-ifremer/tuna_mediterranean/tags/formatted/281B-4949/tagging_event.csv
Uploaded → s3://gfts-ifremer/tuna_mediterranean/tags/formatted/281B-4949/metadata.json

Done. Load with:
  load_tag(tag_root="s3://gfts-ifremer/tuna_mediterranean/tags/formatted", tag_name="281B-4949")
